# Proyecto 3: Galaxy Scaling Relations

Santiago Andrés Acosta Díaz

## Código para el menú

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
import sqlite3
import pandas as pd

# Define the checkboxes and their associated strings.
# You can assign the same string to multiple checkboxes.
checkbox_data = [
    ("Detection method distribution", ["discoverymethod"]),
    ("Period-radius diagram", ["pl_orbper", "pl_rade"]),
    ("Mass-radius relation", ["pl_bmasse", "pl_rade"]),   # shares string with Option A
    ("Equilibrium temperature histogram", ["pl_eqt"]),
    (r"$R_p$ by host star type", ["pl_rade", "st_spectype"]),   # shares string with Option B
    ("Discovery timeline", ["disc_year", "discoverymethod"]),   # shares string with Option B
]

# Create checkboxes
checkboxes = []
for label, string in checkbox_data:
    cb = widgets.Checkbox(
        value=False,
        description=label,
        indent=False
    )

    checkboxes.append((cb, string))

# Container to display the current list
output = widgets.Output()

unique_list = ["pl_name"]

# Function to rebuild the unique list from current checkbox states
def update_list(change=None):
    global unique_list
    # Count active checkboxes per string
    active_counts = {}
    for cb, items in checkboxes:
        for string in items:
            if cb.value:
                active_counts[string] = active_counts.get(string, 0) + 1
    
    # Build the list: include a string only if its count > 0
    unique_list = ["pl_name"] + [s for s, count in active_counts.items() if count > 0]
    
    # Update the output area
    with output:
        clear_output(wait=True)
        print("Current query:", unique_list)


# Attach the update function to every checkbox
for cb, _ in checkboxes:
    cb.observe(update_list, names='value')

# Replace the "checkbox_grid" section with this:
row1 = widgets.HBox(
    [checkboxes[0][0], checkboxes[1][0], checkboxes[2][0]],
    layout=widgets.Layout(gap='30px')  # spacing between items
)
row2 = widgets.HBox(
    [checkboxes[3][0], checkboxes[4][0], checkboxes[5][0]],
    layout=widgets.Layout(gap='30px'))




# FILTER BUTTON

# Global variable to store the checkbox state
filterflag = False

# Create the checkbox
extra_checkbox = widgets.Checkbox(
    value=False,
    description="Apply filter",
    indent=False
)

# Observer to update the global variable
def update_global_flag(change):
    global filterflag
    filterflag = change['new']

extra_checkbox.observe(update_global_flag, names='value')


# BOTONES
button1 = widgets.Button(description="Query")
button2 = widgets.Button(description="Save Database")

# TEXTO DE GUARDADO

savepath = ""

# Create the text widget
text_input = widgets.Text(
    value='',
    placeholder='Guardar como...',
    description='',
    disabled=False,
    layout=widgets.Layout(width='300px')   # optional width
)

# Observer: updates the global variable on every keystroke
def on_text_change(change):
    global savepath
    savepath = change['new']

text_input.observe(on_text_change, names='value')



# La base de datos como tal
result = None


# FUNCIONES DE LOS BOTONES

def action1(b):
    
    with output:
        global result
        
        clear_output(wait=True)
        print("Relizando query con parámetros:", unique_list)
        
        if filterflag:
            print("Filtrando filas NULL")
            null_checks = " AND ".join([f"{col} IS NOT NULL" for col in unique_list])
            result = NasaExoplanetArchive.query_criteria(table="pscomppars", select=unique_list, where=null_checks )
        else: 
            result = NasaExoplanetArchive.query_criteria(table="pscomppars", select=unique_list)

        print(f"Resultados del query: {len(result)} resultados")


def action2(b):
    with output:
        global savepath
        
        clear_output(wait=True)
        if savepath == "":
            print("Añadir nombre al archivo")
        else:
            if result == None:
                print("No hay datos")
            else:
                # Convert to Pandas DataFrame
                df = result.to_pandas()
                
                # Create a connection to a SQLite database (it will be created if not exists)
                names = savepath.split(".")

                conn = sqlite3.connect(f'{names[0]}.db')
                
                # Write the DataFrame to a SQLite table (replace 'my_table' with your desired table name)
                df.to_sql('my_table', conn, if_exists='replace', index=False)
                
                # Close the connection
                conn.close()

                print(f"Se guardó la tabla como {names[0]}.db")
        
button1.on_click(action1)
button2.on_click(action2)




# Third row: empty placeholder in column 1, then the two buttons

row3 = widgets.HBox(
    [extra_checkbox, button1, button2, text_input],
    layout=widgets.Layout(gap='30px')
)

# -------------------------------------------------------------------
# 5. Combine and display
# -------------------------------------------------------------------
ui = widgets.VBox([row1, row2, row3, output])


<string>:24: FutureWarning: tag.strict is not set. Currently defaults to False (permissive tag matching). In a future major version the default will change to True (require tags to contain a dot). Set tag.strict = true or tag.strict = false explicitly in your [tool.setuptools_scm] / [tool.vcs-versioning] config to silence this warning.


# Menú



## Elegir las gráficas que se van a realizar

Aquí elegimos, a priori, las columnas que vamos a utilizar en el query

In [2]:
display(ui)

## Opciones de filtro

Aquí vamos a seleccionar filtros sencillos

## Hacemos el query

In [29]:
# Build the null-check condition dynamically
null_checks = " AND ".join([f"{col} IS NOT NULL" for col in unique_list])
result = NasaExoplanetArchive.query_criteria(table="pscomppars", select=unique_list, where=null_checks )

In [32]:
result.size()

AttributeError: 'QTable' object has no attribute 'size'